In [2]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [9]:
p = 40
a = 2
h = 19

korn =2 #su2

topologyfile = f"evaltopologyfiles/df_p{p}_a{a}_h{h}.edgelist"
serverfile = f"evalserverfiles/df_p{p}_a{a}_h{h}.sv"
numsw = a*(a*h+1)
npfile = f"evalnetpathfiles/netpath_df_p{p}_a{a}_h{h}_su{korn}.np"


stime = 144 # ms
g = a*h+1
nlinks = a*(a-1)*g + g*(g-1) # uni-directional
nhosts = a*p*g
bw = 1342176000 # B per second
load_list = [1,2,3,4,6,7,8,9]
seed_list = [1,2,3,4,5]
topologytype = 3
nswitches = a*g
k = p+h+a-1 # nports
os = 1 #unused
nintervals = 8

failpct_list = range(2,11,2)
failseed_list = range(10)

print(f"#switches: {numsw}, #servers: {nhosts}, #links: {nlinks}, #groups: {g}, #ports: {k}")

#switches: 78, #servers: 3120, #links: 1560, #groups: 39, #ports: 60


In [11]:
# generate linkfailurefiles
numtotalbilinks = nlinks//2
with open(f"{homedir}experiments/nsdi26fall/eval_failure_link/df2_lffiles.conf", 'w') as f:
    for failpct in failpct_list:
        numfaillinks = int(nlinks * failpct / 100)
        for failseed in failseed_list:
            linkfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_link/linkfailurefiles/df2_{numfaillinks}_{failseed}.lf"
            f.write(f"python3 {homedir}generate_dring_linkfailurefiles.py --numfaillinks {numfaillinks} --rseed {failseed} --linkfailurefile {linkfailurefile} --graphfile {homedir}{topologyfile} --numtotalbilinks {numtotalbilinks}\n")

(current dir: ~/DRing/src/emp/datacenter/experiments/nsdi26fall/eval_failure_link/)
python3 ../../../pararun.py --conf df2_lffiles.conf --worker 20 